# SAGE-Lite B2: Phase 3 — Router Hidden Dim Screening (H32 vs H64 vs H128)
**Architecture:** Canonical D4 K2 (ConvNeXtV2-Femto + ViT-Tiny 4 layers, P3-C ASDW, top_k = 2)
**Branch:** `crack500-audit` | **Hardware Target:** Google Colab Tesla T4 (16GB VRAM)

### Protocol Overview:
- **H64 Baseline (Reuse):** Đã hoàn tất (`P3_C_D4_K2_best_model_b2_global.pth`, Peak S2 Val Dice = **0.7618**, Loss 0.9518, IoU 0.6386). Không cần train lại.
- **Run 7 (H32):** `router_hidden_dim = 32`, config `configs/p3_ablation/b2_p3_run_c_d4_k2_h32.yaml`
- **Run 9 (H128):** `router_hidden_dim = 128`, config `configs/p3_ablation/b2_p3_run_c_d4_k2_h128.yaml`
- **Full Automated Lifecycle:** Train $\to$ Routing Diagnostics $\to$ Error Analysis $\to$ Zip & Export $\to$ Comparative Metrics Table.

## 1. Setup Environment & Repository Synchronization

In [ ]:
# Cell 1: Hardware & CUDA Verification
!nvidia-smi

In [ ]:
# Cell 2: Google Drive Mount & Repository Synchronization
import os
from google.colab import drive

drive.mount('/content/drive')

%cd /content
if not os.path.exists('sage_lite'):
    !git clone -b crack500-audit https://github.com/bach0823/SAGE_LITE.git sage_lite
else:
    %cd /content/sage_lite
    !git fetch origin crack500-audit
    !git checkout crack500-audit
    !git pull origin crack500-audit

%cd /content/sage_lite
!pip install -q -e .
!pip install -q albumentations timm scipy scikit-learn seaborn matplotlib pandas

In [ ]:
# Cell 3: Dataset Invariant Verification
import os

dataset_root = '/content/dataset/Crack500'
if not os.path.exists(dataset_root):
    print(f'[Dataset] {dataset_root} not found! Extracting/preparing Crack500...')
    !python prepare_data/prepare_crack500.py
else:
    print(f'[Dataset] Found canonical dataset at {dataset_root}')

assert os.path.exists(os.path.join(dataset_root, 'train', 'images')), 'Train images missing!'
assert os.path.exists(os.path.join(dataset_root, 'val', 'images')), 'Val images missing!'
print('[Dataset] Verification PASS: Train and Val splits confirmed ready.')

## 2. RUN 7: Train H32 (`router_hidden_dim: 32`)
Cấu hình: `b2_p3_run_c_d4_k2_h32.yaml` — 35 epochs (17 S1 + 18 S2, BS14, D4, K2).

In [ ]:
# Cell 4: Train H32
%cd /content/sage_lite
!python scripts/train_crack.py --config configs/p3_ablation/b2_p3_run_c_d4_k2_h32.yaml --two-stage

## 3. Diagnose & Zip H32
Chạy trọn gói Routing Diagnostics (`full_val`) và Error Analysis (`error_analysis`), sau đó nén toàn bộ artifacts thành tệp zip chuẩn.

In [ ]:
# Cell 5: Full Diagnostics Suite for H32
%cd /content/sage_lite

h32_dir = "/content/drive/MyDrive/crack_seg/P3_C_Canonical_Base_D4_K2_H32"
h32_ckpt = os.path.join(h32_dir, "best_model_b2_global.pth")
h32_diag_val = os.path.join(h32_dir, "diagnostics", "full_val")
h32_diag_err = os.path.join(h32_dir, "diagnostics", "error_analysis")
h32_cfg = "configs/p3_ablation/b2_p3_run_c_d4_k2_h32.yaml"

print("[H32 Diagnostics] 1/2: Running analyze_routing.py (Routing Statistics)...\n")
!python tools/analyze_routing.py \
    --config {h32_cfg} \
    --checkpoint {h32_ckpt} \
    --output_dir {h32_diag_val} \
    --split val

print("\n[H32 Diagnostics] 2/2: Running run_p3_c_error_analysis.py (Error Analysis & Morphology)...\n")
!python tools/run_p3_c_error_analysis.py \
    --config {h32_cfg} \
    --checkpoint {h32_ckpt} \
    --routing-json {h32_diag_val}/routing_statistics.json \
    --output-dir {h32_diag_err} \
    --data-root /content/dataset/Crack500

print("\n[H32 Diagnostics] COMPLETED SUCCESSFULLY!")

In [ ]:
# Cell 6: Zip Full H32 Artifacts (Best/Last Models, Logs, Diagnostics)
%cd /content/drive/MyDrive/crack_seg
!zip -r P3_C_Canonical_Base_D4_K2_H32_Full.zip P3_C_Canonical_Base_D4_K2_H32
print("\n[Archive] Successfully generated: /content/drive/MyDrive/crack_seg/P3_C_Canonical_Base_D4_K2_H32_Full.zip")

## 4. RUN 9: Train H128 (`router_hidden_dim: 128`)
Cấu hình: `b2_p3_run_c_d4_k2_h128.yaml` — 35 epochs (17 S1 + 18 S2, BS14, D4, K2).

In [ ]:
# Cell 7: Train H128
%cd /content/sage_lite
!python scripts/train_crack.py --config configs/p3_ablation/b2_p3_run_c_d4_k2_h128.yaml --two-stage

## 5. Diagnose & Zip H128
Chạy trọn gói Routing Diagnostics (`full_val`) và Error Analysis (`error_analysis`), sau đó nén toàn bộ artifacts thành tệp zip chuẩn.

In [ ]:
# Cell 8: Full Diagnostics Suite for H128
%cd /content/sage_lite

h128_dir = "/content/drive/MyDrive/crack_seg/P3_C_Canonical_Base_D4_K2_H128"
h128_ckpt = os.path.join(h128_dir, "best_model_b2_global.pth")
h128_diag_val = os.path.join(h128_dir, "diagnostics", "full_val")
h128_diag_err = os.path.join(h128_dir, "diagnostics", "error_analysis")
h128_cfg = "configs/p3_ablation/b2_p3_run_c_d4_k2_h128.yaml"

print("[H128 Diagnostics] 1/2: Running analyze_routing.py (Routing Statistics)...\n")
!python tools/analyze_routing.py \
    --config {h128_cfg} \
    --checkpoint {h128_ckpt} \
    --output_dir {h128_diag_val} \
    --split val

print("\n[H128 Diagnostics] 2/2: Running run_p3_c_error_analysis.py (Error Analysis & Morphology)...\n")
!python tools/run_p3_c_error_analysis.py \
    --config {h128_cfg} \
    --checkpoint {h128_ckpt} \
    --routing-json {h128_diag_val}/routing_statistics.json \
    --output-dir {h128_diag_err} \
    --data-root /content/dataset/Crack500

print("\n[H128 Diagnostics] COMPLETED SUCCESSFULLY!")

In [ ]:
# Cell 9: Zip Full H128 Artifacts (Best/Last Models, Logs, Diagnostics)
%cd /content/drive/MyDrive/crack_seg
!zip -r P3_C_Canonical_Base_D4_K2_H128_Full.zip P3_C_Canonical_Base_D4_K2_H128
print("\n[Archive] Successfully generated: /content/drive/MyDrive/crack_seg/P3_C_Canonical_Base_D4_K2_H128_Full.zip")

## 6. Quick Cross-Comparison (H32 vs H64 vs H128)

In [ ]:
# Cell 10: Comparative Summary Table for Phase 3 Decision Gate
import json
import os
import torch
import pandas as pd

runs = {
    'H32 (dim=32)': {
        'ckpt': '/content/drive/MyDrive/crack_seg/P3_C_Canonical_Base_D4_K2_H32/best_model_b2_global.pth',
        'err_json': '/content/drive/MyDrive/crack_seg/P3_C_Canonical_Base_D4_K2_H32/diagnostics/error_analysis/error_summary.json'
    },
    'H64 (Baseline)': {
        'dice': 0.7618, 'loss': 0.9518, 'iou': 0.6386, 'status': 'Canonical Baseline'
    },
    'H128 (dim=128)': {
        'ckpt': '/content/drive/MyDrive/crack_seg/P3_C_Canonical_Base_D4_K2_H128/best_model_b2_global.pth',
        'err_json': '/content/drive/MyDrive/crack_seg/P3_C_Canonical_Base_D4_K2_H128/diagnostics/error_analysis/error_summary.json'
    }
}

summary = []
for name, data in runs.items():
    if name == 'H64 (Baseline)':
        summary.append({'Config': name, 'Best Val Dice': 0.7618, 'Best Val Loss': 0.9518, 'Val IoU': 0.6386, 'Status': 'Reused'})
    else:
        row = {'Config': name, 'Best Val Dice': None, 'Best Val Loss': None, 'Val IoU': None, 'Status': 'Missing'}
        if os.path.exists(data['ckpt']):
            c = torch.load(data['ckpt'], map_location='cpu', weights_only=False)
            row['Best Val Dice'] = round(float(c.get('best_dice', 0.0)), 4)
            row['Best Val Loss'] = round(float(c.get('best_loss', 0.0)), 4)
            row['Status'] = f"Epoch {c.get('epoch', '?')} (Stage {c.get('stage', '?')})"
        if os.path.exists(data['err_json']):
            with open(data['err_json']) as f:
                ej = json.load(f)
                row['Val IoU'] = round(float(ej.get('overall_metrics', {}).get('mean_iou', 0.0)), 4)
        summary.append(row)

df = pd.DataFrame(summary)
print("\n" + "="*75)
print("PHASE 3: ROUTER HIDDEN DIM COMPARATIVE SUMMARY (Validation 348 Samples)")
print("="*75)
print(df.to_string(index=False))
print("="*75 + "\n")